# Домашнє завдання 5: CTE, recursive CTE і Materialized View на Olist

**Датасет.** [Olist Brazilian E-commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) —
близько 100 тис. замовлень бразильського e-commerce-майданчика Olist за 2016–2018 роки.
Використано **5 пов'язаних таблиць**: клієнти, замовлення, позиції замовлень, товари, продавці.

**Роль і сценарій.** Працюємо як Data / ML Engineer: треба підготувати **customer-level feature
store** для моделей прогнозу поведінки клієнтів. Результат — не один великий запит, а
відтворюваний pipeline: typed-схема поверх raw-шару, трансформація розбита на named CTE-stages,
матеріалізація ознак і вимірювання вартості обчислення.

**Наскрізна тема цього ДЗ — вартість обчислення.** Один і той самий набір ознак можна отримати
трьома способами: переобчислити pipeline щоразу, покласти його у `VIEW` (той самий план, просто
схований за назвою) або матеріалізувати у `MATERIALIZED VIEW` і платити за це storage і
застарілими даними. У роботі всі три варіанти створені **на одному й тому самому SQL-тексті**, а
`EXPLAIN (ANALYZE, BUFFERS)` показує різницю в числах.

**Що в цьому notebook:**

| Розділ | Зміст |
|---|---|
| Завдання 1 | `pgserver`, відтворюване отримання Olist, raw-рівень через `COPY`, typed-схема з PK/FK/CHECK, smoke-тести |
| Завдання 2 | CTE-chain із 8 named stages: RFM, cohort quarter, rolling 30d AOV, cancellation rate, preferred category |
| Завдання 3 | recursive CTE на синтетичному referral-дереві: `depth`, `path`, два запобіжники від циклів |
| Завдання 4 | `MATERIALIZED VIEW` + `UNIQUE INDEX` + `REFRESH ... CONCURRENTLY` і демонстрація freshness |
| Завдання 5 | три `EXPLAIN (ANALYZE, BUFFERS)`, зведення таймінгів і Reflection |
| Автоперевірки | `assert`-и, які падають, якщо будь-який артефакт ДЗ відсутній або розійшовся |

**Відтворюваність.** Notebook розрахований на `Restart & Run All`: усі DDL починаються з
`DROP ... IF EXISTS`, `reference_date` береться з даних (а не `CURRENT_DATE`), у синтетичних
даних немає `RANDOM()` і `NOW()`. Якщо Olist недоступний, автоматично вмикається детермінована
stub-схема, і всі SQL-клітинки залишаються тими самими.

---

## Завдання 1. Setup і typed-схема Olist

### 1.1 Середовище PostgreSQL

In [1]:
# ============================================================
# PostgreSQL + Python environment
# ============================================================

%pip install -q pgserver psycopg2-binary "sqlalchemy>=2.0" pandas jupysql kagglehub tzdata

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# Imports and configuration
# ============================================================

import gzip
import io
import os
import shutil
import sys
import tempfile
import time
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = BASE_DIR / "data"
OLIST_DIR = DATA_DIR / "olist"
OLIST_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_colwidth", 70)


def ensure_pg_timezone_files():
    """Windows-колесо pgserver не містить share/postgresql/timezone.

    Без цього каталогу сервер стартує з TimeZone=GMT, а SET TIME ZONE 'UTC' падає.
    У Colab/Linux каталог уже є, тому функція нічого не робить.
    """
    tz_dir = Path(pgserver.__file__).parent / "pginstall" / "share" / "postgresql" / "timezone"
    if tz_dir.exists():
        return False
    import tzdata

    shutil.copytree(Path(tzdata.__file__).parent / "zoneinfo", tz_dir)
    return True


# ============================================================
# Start PostgreSQL through pgserver
# ============================================================

patched_tz = ensure_pg_timezone_files()
CLUSTER_DIR = Path(tempfile.gettempdir()) / "hw5_pg"

pg = pgserver.get_server(str(CLUSTER_DIR), cleanup_mode="stop")

# SQLAlchemy 2.1 віддає "postgresql://" драйверу psycopg (v3); фіксуємо psycopg2 явно,
# бо саме його cursor.copy_expert() використовується для COPY нижче.
engine = create_engine(
    pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1),
    connect_args={"options": "-c timezone=UTC"},
    future=True,
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())
    print("Timezone:", conn.execute(text("SELECT current_setting('TimeZone')")).scalar_one())

print("Google Colab:", IN_COLAB)
print("Каталог даних:", OLIST_DIR)
print("Кластер PostgreSQL:", CLUSTER_DIR)
print("Скопійовано timezone-файли для Windows:", patched_tz)

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit
Timezone: UTC
Google Colab: False
Каталог даних: C:\Work\GoIT\Relational Databases\lesson_5\data\olist
Кластер PostgreSQL: C:\Users\Asus\AppData\Local\Temp\hw5_pg
Скопійовано timezone-файли для Windows: False


SQL виконуємо магією `%%sql` (пакет **JupySQL**), результат повертається як `pandas.DataFrame`.
Python-клітинки залишаються тільки для того, що SQL зробити не може: старт сервера, завантаження
CSV, `REFRESH ... CONCURRENTLY` поза транзакцією, друк планів і автоперевірки.

In [3]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 1
%sql engine

%sql SELECT current_setting('TimeZone') AS session_timezone, NOW() AS server_now;

1 rows affected.

,session_timezone,server_now
0,UTC,2026-10-04 16:22:37.904119+00:00


### 1.2 Відтворюване отримання файлів Olist

Три джерела за пріоритетом, перше доступне виграє:

1. **KaggleHub** — лише якщо креденшели справді налаштовані (`KAGGLE_USERNAME`/`KAGGLE_KEY` або
   `~/.kaggle/kaggle.json`). Перевірка обовʼязкова: без токена `kagglehub` намагається запитати
   логін інтерактивно, а це зависання при *Restart & Run All*.
2. **Локальна папка `data/olist/`** — 5 файлів `*.csv.gz`, які лежать у репозиторії.
3. **Публічне дзеркало HuggingFace** — якщо репозиторій клоновано без даних.

Якщо не спрацювало жодне, `USE_STUB = True` і далі працює детермінована stub-схема з умови ДЗ.

In [4]:
OLIST_FILES = {
    "customers":   "olist_customers_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
}
MIRROR_URL = "https://huggingface.co/datasets/bulutttt/olist-raw-data/resolve/main/"


def _kaggle_credentials_ready():
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    return (Path.home() / ".kaggle" / "kaggle.json").exists()


def resolve_olist_files():
    """Повертає (dict назва -> шлях, опис джерела) або кидає виняток."""
    # 1. Kaggle
    if _kaggle_credentials_ready():
        try:
            import kagglehub

            kaggle_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
            paths = {k: kaggle_dir / v for k, v in OLIST_FILES.items()}
            if all(p.exists() for p in paths.values()):
                return paths, "kagglehub: olistbr/brazilian-ecommerce"
        except Exception as exc:  # немає мережі / токен відкликано
            print("KaggleHub недоступний, переходимо до наступного джерела:", exc)

    # 2. Локальні файли репозиторію
    paths = {}
    for key, name in OLIST_FILES.items():
        gz_path, csv_path = OLIST_DIR / (name + ".gz"), OLIST_DIR / name
        paths[key] = gz_path if gz_path.exists() else csv_path
    if all(p.exists() for p in paths.values()):
        return paths, f"локальна папка {OLIST_DIR}"

    # 3. Публічне дзеркало
    for key, name in OLIST_FILES.items():
        dst = OLIST_DIR / (name + ".gz")
        if not dst.exists():
            print("завантаження", name)
            with urlopen(MIRROR_URL + name, timeout=300) as src, gzip.open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
        paths[key] = dst
    return paths, "публічне дзеркало HuggingFace (bulutttt/olist-raw-data)"


try:
    OLIST_PATHS, OLIST_SOURCE = resolve_olist_files()
    USE_STUB = False
except Exception as exc:
    OLIST_PATHS, USE_STUB = None, True
    OLIST_SOURCE = f"детермінована stub-схема з умови ДЗ ({type(exc).__name__}: {exc})"

print("Джерело даних:", OLIST_SOURCE)
print("Stub-схема:", USE_STUB)
if OLIST_PATHS:
    for key, path in OLIST_PATHS.items():
        print(f"  {key:12s} {path.name:40s} {path.stat().st_size / 1e6:6.2f} МБ")

Джерело даних: локальна папка C:\Work\GoIT\Relational Databases\lesson_5\data\olist
Stub-схема: False
  customers    olist_customers_dataset.csv.gz             4.79 МБ
  orders       olist_orders_dataset.csv.gz                6.93 МБ
  order_items  olist_order_items_dataset.csv.gz           6.62 МБ
  products     olist_products_dataset.csv.gz              1.05 МБ
  sellers      olist_sellers_dataset.csv.gz               0.08 МБ


### 1.3 Raw-рівень

Raw-таблиці `olist_*_raw` дзеркалять CSV **як є**, з типами, які виводить `pandas`, і з
оригінальними назвами колонок — включно з опискою `product_name_lenght` /
`product_description_lenght` у файлі Olist. Вони потрібні лише як проміжний шар і не замінюють
typed-схему з обмеженнями.

Завантаження — через `COPY ... FROM STDIN` (`psycopg2.cursor.copy_expert`), а не
`DataFrame.to_sql`: `to_sql` шле рядки по одному й на 112 тис. позицій займає хвилини замість
секунд.

Stub-схема (fallback) наповнює **ті самі** `olist_*_raw` таблиці детермінованим набором з умови —
3 продавці, 7 товарів, 30 клієнтів, 50 замовлень, 100 позицій. Тому точка розгалуження в
notebook одна, а всі наступні SQL-клітинки однакові в обох режимах.

In [5]:
RAW_TABLES = {
    "customers":   "olist_customers_raw",
    "orders":      "olist_orders_raw",
    "order_items": "olist_order_items_raw",
    "products":    "olist_products_raw",
    "sellers":     "olist_sellers_raw",
}


def load_raw(df, table):
    """DDL через to_sql(head(0)), дані — через COPY ... FROM STDIN."""
    df.head(0).to_sql(table, engine, if_exists="replace", index=False)

    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)

    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur:
            cur.copy_expert(f'COPY "{table}" FROM STDIN WITH (FORMAT csv)', buffer)
        raw_conn.commit()
    finally:
        raw_conn.close()


def build_stub_frames():
    """Детермінований stub з умови ДЗ у вигляді raw-CSV кадрів (без RANDOM/NOW)."""
    day = pd.Timedelta(days=1)
    base = pd.Timestamp("2024-01-01")

    sellers = pd.DataFrame({
        "seller_id": ["s_01", "s_02", "s_03"],
        "seller_zip_code_prefix": [10001, 20001, 30001],
        "seller_city": ["Sao Paulo", "Rio de Janeiro", "Belo Horizonte"],
        "seller_state": ["SP", "RJ", "MG"],
    })
    products = pd.DataFrame({
        "product_id": [f"p_0{i}" for i in range(1, 8)],
        "product_category_name": ["electronics", "electronics", "home", "home",
                                  "fashion", "beauty", "sport"],
        "product_name_lenght": [10, 12, 8, 9, 7, 6, 9],
        "product_description_lenght": [100, 120, 90, 80, 60, 50, 70],
        "product_photos_qty": [2, 3, 1, 2, 1, 1, 2],
        "product_weight_g": [800, 1200, 1500, 700, 300, 200, 900],
        "product_length_cm": [20, 25, 30, 18, 10, 8, 20],
        "product_height_cm": [10, 12, 20, 10, 5, 4, 15],
        "product_width_cm": [15, 18, 25, 14, 8, 6, 16],
    })
    n_cust = range(1, 31)
    customers = pd.DataFrame({
        "customer_id": [f"c_{n:03d}" for n in n_cust],
        "customer_unique_id": [f"u_{(n - 1) % 25 + 1:03d}" for n in n_cust],
        "customer_zip_code_prefix": [10000 + n for n in n_cust],
        "customer_city": [f"City{n % 10}" for n in n_cust],
        "customer_state": [["SP", "RJ", "MG", "BA", "PR"][n % 5] for n in n_cust],
    })
    n_ord = range(1, 51)
    purchase = [base + n * 7 * day for n in n_ord]
    orders = pd.DataFrame({
        "order_id": [f"o_{n:03d}" for n in n_ord],
        "customer_id": [f"c_{(n % 30) + 1:03d}" for n in n_ord],
        "order_status": [["delivered", "delivered", "delivered", "canceled", "shipped"][n % 5]
                         for n in n_ord],
        "order_purchase_timestamp": purchase,
        "order_approved_at": [ts + 1 * day for ts in purchase],
        "order_delivered_carrier_date": [ts + 3 * day for ts in purchase],
        "order_delivered_customer_date": [ts + 8 * day for ts in purchase],
        "order_estimated_delivery_date": [ts + 10 * day for ts in purchase],
    })
    n_item = range(1, 101)
    order_items = pd.DataFrame({
        "order_id": [f"o_{(n - 1) % 50 + 1:03d}" for n in n_item],
        "order_item_id": [(n - 1) // 50 + 1 for n in n_item],
        "product_id": [f"p_0{(n % 7) + 1}" for n in n_item],
        "seller_id": [f"s_0{(n % 3) + 1}" for n in n_item],
        "shipping_limit_date": [base + (n * 7 + 5) * day for n in n_item],
        "price": [float(n % 200 + 10) for n in n_item],
        "freight_value": [float(n % 30 + 5) for n in n_item],
    })
    return {"customers": customers, "orders": orders, "order_items": order_items,
            "products": products, "sellers": sellers}


frames = build_stub_frames() if USE_STUB else {k: pd.read_csv(p) for k, p in OLIST_PATHS.items()}

for key, table in RAW_TABLES.items():
    load_raw(frames[key], table)
    print(f"{table:26s} {frames[key].shape[0]:>7,} рядків x {frames[key].shape[1]} колонок")

olist_customers_raw         99,441 рядків x 5 колонок


olist_orders_raw            99,441 рядків x 8 колонок


olist_order_items_raw      112,650 рядків x 7 колонок
olist_products_raw          32,951 рядків x 9 колонок
olist_sellers_raw            3,095 рядків x 4 колонок


**Висновок 1.3.** Raw-шар завантажено з локальних `*.csv.gz`: **99 441** клієнт, **99 441**
замовлення, **112 650** позицій, **32 951** товар, **3 095** продавців. Позицій більше, ніж
замовлень, у 1.13 раза — це і є та різниця грануляності, через яку `stage1` зводить позиції до
замовлення **перед** будь-якою агрегацією по клієнту.

Перші рядки raw-рівня — саме те, що лежить у CSV, без приведення типів:

In [6]:
%%sql
SELECT order_id, order_item_id, product_id, seller_id, price, freight_value
FROM olist_order_items_raw
ORDER BY order_id, order_item_id
LIMIT 5;

5 rows affected.

,order_id,order_item_id,product_id,seller_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90,18.14


### 1.4 Typed-схема: ключі й обмеження цілісності

Typed-схема — це контракт даних, і саме вона використовується в усіх наступних завданнях.
`DROP ... CASCADE` на початку робить клітинку ідемпотентною, тому *Restart & Run All* не вимагає
ручного прибирання. Разом із таблицями знімаємо й похідні обʼєкти ДЗ (`VIEW`, `MATERIALIZED
VIEW`, `hw5_referrals`) — інакше `CASCADE` зніс би їх пізніше в непередбачуваному порядку.

In [7]:
%%sql
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
DROP VIEW IF EXISTS v_olist_customer_features CASCADE;
DROP TABLE IF EXISTS hw5_referrals     CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

SELECT COUNT(*) AS typed_objects_left
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN ('olist_customers', 'olist_orders', 'olist_order_items',
                     'olist_products', 'olist_sellers', 'hw5_referrals',
                     'v_olist_customer_features', 'mv_olist_customer_features');

1 rows affected.

,typed_objects_left
0,0


`typed_objects_left = 0` означає, що клітинка нижче будує схему з нуля, а не доливає дані в
залишки попереднього прогону.

In [8]:
%%sql
CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT    NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT      NOT NULL REFERENCES olist_customers (customer_id),
    order_status                  TEXT      NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN ('created', 'approved', 'invoiced', 'processing',
                            'shipped', 'delivered', 'unavailable', 'canceled'))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT    NOT NULL REFERENCES olist_orders (order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products (product_id),
    seller_id           TEXT REFERENCES olist_sellers (seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name LIKE 'olist_%'
  AND table_name NOT LIKE '%_raw'
ORDER BY table_name;

5 rows affected.

,table_name
0,olist_customers
1,olist_order_items
2,olist_orders
3,olist_products
4,olist_sellers


### 1.5 Перенесення raw → typed

Приведення типів робиться явним `INSERT ... SELECT`: текст із CSV → `INTEGER`, `TIMESTAMP`,
`NUMERIC(12,2)`, `CHAR(2)`. Тут же нормалізуються назви колонок: у CSV Olist вони написані з
опискою (`product_name_lenght`), у typed-схемі — правильно (`product_name_length`).

Порядок вставки визначається зовнішніми ключами: спочатку `customers`, `products`, `sellers`,
потім `orders`, і лише потім `order_items`.

In [9]:
%%sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

SELECT
    (SELECT COUNT(*) FROM olist_customers)   AS customers,
    (SELECT COUNT(*) FROM olist_products)    AS products,
    (SELECT COUNT(*) FROM olist_sellers)     AS sellers,
    (SELECT COUNT(*) FROM olist_orders)      AS orders,
    (SELECT COUNT(*) FROM olist_order_items) AS order_items;

99441 rows affected.

32951 rows affected.

3095 rows affected.

99441 rows affected.

112650 rows affected.

1 rows affected.

,customers,products,sellers,orders,order_items
0,99441,32951,3095,99441,112650


### 1.6 Smoke-тести

Три перевірки: чи не втрачено рядки на переході raw → typed, які саме обмеження створено, і яка
вийшла `reference_date` для recency.

In [10]:
%%sql
SELECT 'customers'   AS table_name,
       (SELECT COUNT(*) FROM olist_customers_raw)   AS raw_rows,
       (SELECT COUNT(*) FROM olist_customers)       AS typed_rows,
       (SELECT COUNT(*) FROM olist_customers_raw) -
       (SELECT COUNT(*) FROM olist_customers)       AS diff
UNION ALL
SELECT 'orders',
       (SELECT COUNT(*) FROM olist_orders_raw),
       (SELECT COUNT(*) FROM olist_orders),
       (SELECT COUNT(*) FROM olist_orders_raw) - (SELECT COUNT(*) FROM olist_orders)
UNION ALL
SELECT 'order_items',
       (SELECT COUNT(*) FROM olist_order_items_raw),
       (SELECT COUNT(*) FROM olist_order_items),
       (SELECT COUNT(*) FROM olist_order_items_raw) - (SELECT COUNT(*) FROM olist_order_items)
UNION ALL
SELECT 'products',
       (SELECT COUNT(*) FROM olist_products_raw),
       (SELECT COUNT(*) FROM olist_products),
       (SELECT COUNT(*) FROM olist_products_raw) - (SELECT COUNT(*) FROM olist_products)
UNION ALL
SELECT 'sellers',
       (SELECT COUNT(*) FROM olist_sellers_raw),
       (SELECT COUNT(*) FROM olist_sellers),
       (SELECT COUNT(*) FROM olist_sellers_raw) - (SELECT COUNT(*) FROM olist_sellers)
ORDER BY table_name;

5 rows affected.

,table_name,raw_rows,typed_rows,diff
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,sellers,3095,3095,0


**Висновок 1.6a.** `diff = 0` для всіх пʼяти таблиць — жоден рядок не втрачено на переході
raw → typed. Це означає, що всі `FOREIGN KEY` і `CHECK` виконались без жодного відкинутого
рядка: у Olist посилання `orders → customers`, `order_items → orders/products/sellers` цілі, а
статуси замовлень не виходять за список із `CHECK`.

In [11]:
%%sql
SELECT
    c.relname AS table_name,
    COUNT(*) FILTER (WHERE con.contype = 'p') AS primary_keys,
    COUNT(*) FILTER (WHERE con.contype = 'f') AS foreign_keys,
    COUNT(*) FILTER (WHERE con.contype = 'c') AS checks
FROM pg_constraint AS con
JOIN pg_class     AS c ON c.oid = con.conrelid
JOIN pg_namespace AS n ON n.oid = c.relnamespace
WHERE n.nspname = 'public'
  AND c.relname IN ('olist_customers', 'olist_orders', 'olist_order_items',
                    'olist_products', 'olist_sellers')
GROUP BY c.relname
ORDER BY c.relname;

5 rows affected.

,table_name,primary_keys,foreign_keys,checks
0,olist_customers,1,0,0
1,olist_order_items,1,3,2
2,olist_orders,1,1,1
3,olist_products,1,0,7
4,olist_sellers,1,0,0


**Висновок 1.6b.** Схема має **5 `PRIMARY KEY`** (по одному на таблицю, у `olist_order_items`
складений — `order_id, order_item_id`), **4 `FOREIGN KEY`** (`orders → customers` і три з
`order_items`) і **10 `CHECK`**: сім на невідʼємні габарити товару, два на `price` та
`freight_value` і один на список статусів замовлення.

In [12]:
%%sql
SELECT
    MIN(order_purchase_timestamp)::DATE      AS first_purchase,
    MAX(order_purchase_timestamp)::DATE      AS last_purchase,
    (MAX(order_purchase_timestamp)::DATE + 1) AS reference_date,
    COUNT(*)                                 AS n_orders,
    COUNT(DISTINCT customer_id)              AS n_customers,
    COUNT(*) FILTER (WHERE order_status = 'canceled') AS n_canceled
FROM olist_orders;

1 rows affected.

,first_purchase,last_purchase,reference_date,n_orders,n_customers,n_canceled
0,2016-09-04,2018-10-17,2018-10-18,99441,99441,625


**Висновок 1.6c.** Історія Olist — з **2016-09-04** до **2018-10-17**, тому стабільна
`reference_date = 2018-10-18`. Якби `recency_days` рахувався від `CURRENT_DATE`, значення
перевищували б 2 700 днів і змінювалися б щодня: ознака залежала б від дати запуску ноутбука,
а не від даних. Скасованих замовлень **625** із 99 441 (0.63 %) — саме з цієї частки виходить
`cancellation_rate`.

---

## Завдання 2. CTE-chain pipeline із 8 named stages

Pipeline свідомо розрізаний на етапи з однією відповідальністю кожен — так само, як це
виглядає в аналітичних SQL/dbt-проєктах. Кожен stage можна перевірити окремо, і кожен має
коментар `-- Stage N: ...`.

| Stage | Відповідальність | Обовʼязкова ознака |
|---|---|---|
| `reference_date` | `MAX(order_purchase_timestamp)::DATE + 1` | стабільна база для recency |
| `stage1_orders_total` | одне замовлення — один рядок, `price + freight_value` | — |
| `stage2_rfm` | `recency_days`, `frequency`, `monetary`, `first_order_date` | **RFM** |
| `stage3_cohort` | `DATE_TRUNC('quarter', first_order_date)` | **cohort_quarter** |
| `stage4_rolling_aov` | window `AVG` у вікні `RANGE ... '30 days' PRECEDING` + `ROW_NUMBER()` | **avg_order_value_30d** |
| `stage5_cancel_rate` | частка `order_status = 'canceled'` | **cancellation_rate** |
| `stage6_category_revenue` | виручка клієнт × категорія | — |
| `stage7_preferred_category` | top-1 категорія за виручкою | **preferred_category** |

Три рішення, які варто назвати окремо:

- **`reference_date` з даних, а не `CURRENT_DATE`.** Olist закінчується 2018-10-17, тож
  `CURRENT_DATE` дав би `recency_days` понад 2 700 днів і нове значення щодня. Ознака має бути
  функцією від даних, а не від дати запуску ноутбука.
- **`stage1` зводить позиції до замовлення до всіх інших агрегацій.** Якби RFM рахувався прямо
  на `order_items`, `frequency` порахувала б позиції, а не замовлення, і роздулась би в
  середньому в 1.13 раза. Це та сама пастка грануляності, що й у ДЗ4.
- **`stage4` не руйнує деталізацію рядків.** Window-функція лишає один рядок на замовлення,
  тому поряд із ковзним середнім доступний `ROW_NUMBER()`, і фінальний `JOIN` бере значення
  станом на **останнє** замовлення клієнта (`rn_recent = 1`).

Фінальний `SELECT` починається з `olist_customers` і зʼєднується через `LEFT JOIN`, щоб клієнти
без замовлень не зникали з feature store, а отримували явні значення-заглушки
(`recency_days = 9999`, `frequency = 0`, `preferred_category = '(none)'`).

Pipeline зберігаємо як звичайний `VIEW`: це дає одну назву для десятка стейджів і згодом —
предметне порівняння `VIEW` vs `MATERIALIZED VIEW` у Reflection.

In [13]:
%%sql
CREATE VIEW v_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: стабільна точка відліку recency — з самих даних, а не CURRENT_DATE.
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: грануляність «одне замовлення — один рядок»: позиції + доставка.
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp       AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2)
                                         AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
           ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM — recency_days від as_of_date, frequency, monetary, перша покупка.
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*)                                     AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2)           AS monetary,
        MIN(s.order_date)                            AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: квартал когорти за датою першого замовлення.
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day AOV без втрати деталізації рядків + маркер останнього замовлення.
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: частка скасованих замовлень клієнта.
    SELECT
        customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4)
            AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: виручка в розрізі клієнт x категорія товару.
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2)
            AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
      ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
           ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: top-1 категорія за виручкою; нічию розвʼязує назва категорії.
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_unique_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999)                     AS recency_days,
    COALESCE(coh.frequency, 0)                           AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2)            AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d,
    COALESCE(cx.cancellation_rate, 0)::NUMERIC(6, 4)     AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)')         AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2)     AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
       ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
       ON aov.customer_id = c.customer_id
      AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cx
       ON cx.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
       ON pc.customer_id = c.customer_id;

""


Повний набір ознак, топ-20 за `monetary`:

In [14]:
%%sql
SELECT *
FROM v_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 20;

20 rows affected.

,customer_id,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,459bef486812aa25204be022145caa62,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,ff4159b92c40ebe40454e3e6a7c35ed6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
...,...,...,...,...,...,...,...,...,...,...,...,...
15,3be2c536886b2ea4668eced3a80dd0bb,5e713be0853d8986528d7869a0811d2b,PA,621,1,4042.74,2017-02-04,2017-01-01,4042.74,0.0000,(unknown),4042.74
16,eb7a157e8da9c488cd4ddc48711f1097,58483a1c055dfb600f57c5b867174542,SP,135,1,4034.44,2018-06-05,2018-04-01,4034.44,0.0000,bebes,4034.44
17,c6695e3b1e48680db36b487419fb0398,011875f0176909c5cf0b14a9138bb691,SP,579,1,4016.91,2017-03-18,2017-01-01,4016.91,0.0000,relogios_presentes,4016.91
18,31e83c01fce824d0ff786fcd48dad009,f0767ae738c3d90e7b737d7b8b8bb4d1,RJ,157,1,3979.55,2018-05-14,2018-04-01,3979.55,0.0000,ferramentas_jardim,3979.55


**Висновок 2.1.** Pipeline дає рівно один рядок на `customer_id` з усіма пʼятьма обовʼязковими
групами ознак. Топ за `monetary` веде клієнт із **13 664.08** (категорія `telefonia_fixa`) —
це одне замовлення, а не накопичена історія. Зверніть увагу на колонку `frequency`: вона
дорівнює `1` у кожному рядку, і це не випадковість — розбір нижче.

Профіль отриманих ознак — покриття, діапазони й клієнти без замовлень:

In [15]:
%%sql
SELECT
    COUNT(*)                                            AS n_rows,
    COUNT(DISTINCT customer_id)                         AS n_customers,
    COUNT(*) FILTER (WHERE frequency = 0)               AS customers_without_orders,
    COUNT(*) FILTER (WHERE frequency > 1)               AS repeat_customers,
    MIN(recency_days)                                   AS min_recency,
    MAX(recency_days) FILTER (WHERE frequency > 0)      AS max_recency_with_orders,
    ROUND(AVG(monetary) FILTER (WHERE frequency > 0), 2) AS avg_monetary,
    ROUND(AVG(cancellation_rate), 4)                    AS avg_cancellation_rate,
    COUNT(*) FILTER (WHERE cancellation_rate > 0)       AS customers_with_cancellation
FROM v_olist_customer_features;

1 rows affected.

,n_rows,n_customers,customers_without_orders,repeat_customers,min_recency,max_recency_with_orders,avg_monetary,avg_cancellation_rate,customers_with_cancellation
0,99441,99441,0,0,1,774,159.33,0.0063,625


**Висновок 2.2.** Покриття повне: **99 441** рядок на **99 441** `customer_id`, клієнтів без
замовлень немає (кожен `customer_id` у Olist створюється разом із замовленням, тому
`LEFT JOIN`-заглушки тут не задіяні — але вони потрібні, щоб pipeline не ламався на даних, де
такі клієнти є). `recency_days` лежить у діапазоні **1…774** дні, середній `monetary` —
**159.33**, середній `cancellation_rate` — **0.0063**, і рівно **625** клієнтів мають
скасування. `repeat_customers = 0` — те саме попередження, що й у 2.1.

Розподіл по когортах (`cohort_quarter`) — перевірка `stage3`:

In [16]:
%%sql
SELECT
    cohort_quarter,
    COUNT(*)                        AS n_customers,
    ROUND(AVG(monetary), 2)         AS avg_monetary,
    ROUND(AVG(frequency), 4)        AS avg_frequency,
    ROUND(AVG(cancellation_rate), 4) AS avg_cancellation_rate
FROM v_olist_customer_features
WHERE cohort_quarter IS NOT NULL
GROUP BY cohort_quarter
ORDER BY cohort_quarter;

10 rows affected.

,cohort_quarter,n_customers,avg_monetary,avg_frequency,avg_cancellation_rate
0,2016-07-01,4,88.69,1.0000,0.5000
1,2016-10-01,325,174.86,1.0000,0.0738
2,2017-01-01,5262,162.58,1.0000,0.0101
3,2017-04-01,9349,160.61,1.0000,0.0067
4,2017-07-01,12642,156.11,1.0000,0.0059
5,2017-10-01,17848,157.55,1.0000,0.0041
6,2018-01-01,21208,153.21,1.0000,0.0063
7,2018-04-01,19979,166.78,1.0000,0.0029
8,2018-07-01,12820,160.86,1.0000,0.0109
9,2018-10-01,4,0.00,1.0000,1.0000


**Висновок 2.3.** Когорти показують зростання майданчика: від **4** клієнтів у 2016-Q3 до
**21 208** у 2018-Q1. Два краї датасету — обрізані: 2016-Q3 має `cancellation_rate = 0.5000`
(два скасування з чотирьох), а 2016-Q4 — **0.0738**, на порядок вище за стабільні
**0.003…0.011** у 2017–2018. Остання когорта 2018-Q4 — це **4** клієнти з
`avg_monetary = 0.00` і `cancellation_rate = 1.0000`: датасет обривається 2018-10-17, і ці
замовлення так і залишились нескасованими-недоставленими. У ML це класичний кандидат на
відсікання крайових когорт, інакше модель вчиться на артефактах вивантаження.

### 2.1 Грануляність ключа: чому `frequency` вийшла рівно 1

У виводі вище `frequency = 1` у кожного рядка, `repeat_customers = 0`, а `avg_frequency`
дорівнює рівно `1.0000` в усіх когортах. Це не помилка pipeline — це властивість Olist:
`customer_id` там видається **на кожне замовлення**, а реального покупця ідентифікує
`customer_unique_id`. Запит нижче показує обидві грануляності поряд.

Для цього ДЗ ключ feature store залишається `customer_id` — так вимагає умова, і саме на нього
створюється `UNIQUE INDEX` для `REFRESH CONCURRENTLY`. Але знати цю властивість обовʼязково: на
`customer_id` ознаки `frequency` і rolling AOV вироджуються, тому в реальній моделі RFM
рахувався б на `customer_unique_id`, а `customer_id` залишався б лише технічним ключем
замовлення.

In [17]:
%%sql
SELECT
    (SELECT COUNT(*) FROM olist_customers)                           AS customer_id_rows,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers) AS real_buyers,
    (SELECT MAX(n) FROM (
        SELECT COUNT(*) AS n FROM olist_orders GROUP BY customer_id
    ) AS t)                                                          AS max_orders_per_customer_id,
    (SELECT MAX(n) FROM (
        SELECT COUNT(*) AS n
        FROM olist_orders AS o
        JOIN olist_customers AS c ON c.customer_id = o.customer_id
        GROUP BY c.customer_unique_id
    ) AS t)                                                          AS max_orders_per_buyer,
    (SELECT COUNT(*) FROM (
        SELECT c.customer_unique_id
        FROM olist_orders AS o
        JOIN olist_customers AS c ON c.customer_id = o.customer_id
        GROUP BY c.customer_unique_id
        HAVING COUNT(*) > 1
    ) AS t)                                                          AS repeat_buyers;

1 rows affected.

,customer_id_rows,real_buyers,max_orders_per_customer_id,max_orders_per_buyer,repeat_buyers
0,99441,96096,1,17,2997


**Висновок 2.4.** **99 441** `customer_id` відповідають лише **96 096** реальним покупцям.
Максимум замовлень на `customer_id` — **1**, на `customer_unique_id` — **17**, а повторних
покупців **2 997**. Тобто на ключі `customer_id` ознака `frequency` тотожно дорівнює одиниці,
`monetary` зводиться до суми одного замовлення, а 30-денне вікно rolling AOV завжди містить
рівно один рядок. Ознаки формально присутні й pipeline коректний, але передбачної сили на цьому
ключі вони не мають — і це властивість датасету, а не SQL.

Окрема перевірка `stage4`. Щоб у вікні було більше одного замовлення, беремо покупця
(`customer_unique_id`) з найдовшою історією — вікно й синтаксис ті самі, що в pipeline.
Колонка `orders_in_window` показує, скільки замовлень реально потрапило у 30-денне вікно:
видно і що деталізація рядків збереглася, і що старіші замовлення з вікна виходять.

In [18]:
%%sql
WITH busiest_buyer AS (
    SELECT c.customer_unique_id
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
    ORDER BY COUNT(*) DESC, c.customer_unique_id
    LIMIT 1
),
orders_total AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp AS order_ts,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2)
            AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    JOIN busiest_buyer  AS b ON b.customer_unique_id = c.customer_unique_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
)
SELECT
    LEFT(order_id, 8) AS order_id_short,
    order_ts,
    order_total,
    AVG(order_total) OVER (
        PARTITION BY customer_unique_id
        ORDER BY order_ts
        RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
    )::NUMERIC(12, 2) AS avg_order_value_30d,
    COUNT(*) OVER (
        PARTITION BY customer_unique_id
        ORDER BY order_ts
        RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
    ) AS orders_in_window
FROM orders_total
ORDER BY order_ts;

17 rows affected.

,order_id_short,order_ts,order_total,avg_order_value_30d,orders_in_window
0,5d848f3d,2017-05-15 23:30:03,22.77,22.77,1
1,36963470,2017-06-18 22:56:48,51.75,51.75,1
2,5837a2c8,2017-07-17 22:11:13,0.00,25.88,2
3,4f62d593,2017-07-18 23:10:58,21.77,10.89,2
4,bf92c69b,2017-07-24 22:11:50,166.71,62.83,3
...,...,...,...,...,...
12,838f8e96,2018-07-04 22:35:13,30.41,30.41,1
13,112eb6f3,2018-07-23 21:53:02,107.85,69.13,2
14,c2213109,2018-08-07 23:32:14,34.44,71.15,2
15,a1a6eada,2018-08-18 12:39:19,96.40,79.56,3


**Висновок 2.5.** На покупці з найдовшою історією window-функція працює так, як задумано:
`orders_in_window` змінюється від замовлення до замовлення, бо при переході по `order_ts` старі
замовлення виходять за межу `INTERVAL '30 days' PRECEDING`, а `avg_order_value_30d` рахується
лише по тих, що залишились у вікні. Головне — рядки не злились: `RANGE BETWEEN ...` дає
агрегат **поряд** із деталізацією, тому в pipeline можна тим самим проходом узяти
`ROW_NUMBER()` і дістати значення станом на останнє замовлення.

Топ-10 `preferred_category` — перевірка `stage6`/`stage7`:

In [19]:
%%sql
SELECT
    preferred_category,
    COUNT(*)                                     AS n_customers,
    ROUND(AVG(preferred_category_revenue), 2)    AS avg_category_revenue,
    ROUND(SUM(preferred_category_revenue), 2)    AS total_category_revenue
FROM v_olist_customer_features
WHERE frequency > 0
GROUP BY preferred_category
ORDER BY n_customers DESC
LIMIT 10;

10 rows affected.

,preferred_category,n_customers,avg_category_revenue,total_category_revenue
0,cama_mesa_banho,9341,132.27,1235531.26
1,beleza_saude,8804,163.48,1439266.61
2,esporte_lazer,7684,150.10,1153381.68
3,informatica_acessorios,6670,158.39,1056463.41
4,moveis_decoracao,6339,140.79,892449.35
5,utilidades_domesticas,5822,132.97,774167.78
6,relogios_presentes,5604,232.60,1303467.35
7,telefonia,4180,94.28,394072.97
8,automotivo,3880,176.39,684380.86
9,brinquedos,3866,144.81,559820.04


**Висновок 2.6.** `preferred_category` розподілена очікувано для Olist: лідирують
`cama_mesa_banho` (**9 341** клієнт), `beleza_saude` (**8 804**) і `esporte_lazer`
(**7 684**). Середній чек по категоріях відрізняється майже в 2.5 раза — від **94.28**
(`telefonia`) до **232.60** (`relogios_presentes`), тому для моделі корисна не лише сама
категорія, а й пара «категорія + її виручка». Нічия в `ROW_NUMBER()` розвʼязується назвою
категорії, тож результат детермінований між прогонами.

---

## Завдання 3. Recursive CTE для referral chain

Olist **не має** referral-таблиці, тому `hw5_referrals` — навчальна імітація referral-графа, а
не колонка датасету. Будуємо її детерміновано на перших `customer_id` у лексикографічному
порядку:

- **дерево** на 15 клієнтах: батько вузла `rn` — це вузол `rn / 2` (цілочисельно), тобто
  бінарне дерево глибиною 4 — 14 ребер;
- **справжній цикл** `A → B → C → A` на трьох інших клієнтах. `PRIMARY KEY
  (referred_customer_id)` цьому не перешкоджає: кожен вузол має рівно одного referrer, і замкнене
  коло цю умову не порушує. Саме на ньому далі перевіряється захист від циклів.

`referred_at` — фіксований `TIMESTAMPTZ`, а не `NOW()`: інакше вивід клітинки змінювався б
щозапуску й ноутбук перестав би бути відтворюваним.

In [20]:
%%sql
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT        NOT NULL REFERENCES olist_customers (customer_id),
    referred_customer_id TEXT        NOT NULL REFERENCES olist_customers (customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT TIMESTAMPTZ '2025-01-01 00:00:00+00',
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT customer_id, ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 18
),
tree_edges AS (
    -- бінарне дерево: батько вузла rn — вузол rn / 2
    SELECT parent.customer_id AS referrer_customer_id,
           child.customer_id  AS referred_customer_id
    FROM ordered_customers AS child
    JOIN ordered_customers AS parent ON parent.rn = child.rn / 2
    WHERE child.rn BETWEEN 2 AND 15
),
cycle_edges AS (
    -- замкнений цикл 16 -> 17 -> 18 -> 16, навмисно, для перевірки захисту
    SELECT a.customer_id AS referrer_customer_id,
           b.customer_id AS referred_customer_id
    FROM ordered_customers AS a
    JOIN ordered_customers AS b
      ON b.rn = CASE WHEN a.rn = 18 THEN 16 ELSE a.rn + 1 END
    WHERE a.rn BETWEEN 16 AND 18
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT * FROM tree_edges
UNION ALL
SELECT * FROM cycle_edges
ON CONFLICT (referred_customer_id) DO NOTHING;

SELECT
    COUNT(*)                                     AS n_edges,
    COUNT(DISTINCT referrer_customer_id)          AS n_referrers,
    COUNT(DISTINCT referred_customer_id)          AS n_referred,
    MIN(referred_at)                              AS referred_at
FROM hw5_referrals;

17 rows affected.

1 rows affected.

,n_edges,n_referrers,n_referred,referred_at
0,17,10,17,2025-01-01 00:00:00+00:00


**Висновок 3.1.** Створено **17** ребер: 14 — бінарне дерево на 15 клієнтах, 3 — замкнений
цикл. `n_referred = 17` при `n_referrers = 10`: один referrer може привести кількох, але кожен
referred має рівно одного referrer — це гарантує `PRIMARY KEY (referred_customer_id)`.
`referred_at` однаковий у всіх рядках і зафіксований константою, а не `NOW()`, щоб вивід
не змінювався при кожному прогоні.

Усі ребра графа (ідентифікатори скорочено до 8 символів для читабельності):

In [21]:
%%sql
SELECT
    LEFT(referrer_customer_id, 8) AS referrer,
    LEFT(referred_customer_id, 8) AS referred,
    EXISTS (
        SELECT 1 FROM hw5_referrals AS up
        WHERE up.referred_customer_id = r.referrer_customer_id
    ) AS referrer_is_also_referred
FROM hw5_referrals AS r
ORDER BY referrer, referred;

17 rows affected.

,referrer,referred,referrer_is_also_referred
0,00012a2c,000161a0,False
1,00012a2c,0001fd61,False
2,000161a0,0002414f,True
3,000161a0,000379cd,True
4,0001fd61,0004164d,True
...,...,...,...
12,000419c5,00072d03,True
13,000419c5,0009a69b,True
14,000bf812,000e9434,True
15,000e9434,000f17e2,True


**Висновок 3.2.** Колонка `referrer_is_also_referred` робить структуру графа видимою: `False`
лише у двох перших рядків — це ребра з єдиного кореня. У решти ребер referrer і сам комусь
referred, тобто це внутрішні рівні дерева. Три останні рядки — цикл: `000bf812 → 000e9434 →
000f17e2 → 000bf812`, і в ньому `referrer_is_also_referred` дорівнює `True` **у всіх трьох**,
бо кореня в циклі немає за визначенням.

### 3.1 Основний обхід: anchor + recursive part + depth + path

- **ANCHOR** — корені графа: клієнти, на яких немає вхідного ребра. Anti-join через
  `NOT EXISTS`, а не `NOT IN`: якщо підзапит колись поверне `NULL`, `NOT IN` дасть порожній
  результат замовчуванням, і anchor «зникне». `NOT EXISTS` такої пастки не має, навіть якщо
  зараз колонка оголошена `NOT NULL`.
- **RECURSIVE part** — крок від referrer до його referred.
- **Termination** тримається на трьох речах: скінченності графа, перевірці
  `NOT (next = ANY(path_ids))` (не ходити туди, де вже були на цьому шляху) і depth guard
  `rd.depth < 50`.
- `path_ids` — повні `customer_id` (на них працює захист), `path_text` — скорочені до 8 символів
  (щоб шлях було видно у виводі).

In [22]:
%%sql
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: корені — клієнти без вхідного referral-ребра (NULL-safe anti-join).
    SELECT
        c.customer_id,
        1                              AS depth,
        ARRAY[c.customer_id]::TEXT[]   AS path_ids,
        LEFT(c.customer_id, 8)::TEXT   AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: крок referrer -> referred; cycle guard + depth guard.
    SELECT
        r.referred_customer_id,
        rd.depth + 1,
        rd.path_ids || r.referred_customer_id,
        rd.path_text || ' -> ' || LEFT(r.referred_customer_id, 8)
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
      ON rd.customer_id = r.referrer_customer_id
    WHERE NOT (r.referred_customer_id = ANY (rd.path_ids))
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*)        AS n_customers,
    MIN(path_text)  AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;

4 rows affected.

,depth,n_customers,sample_path
0,1,99424,00012a2c
1,2,2,00012a2c -> 000161a0
2,3,4,00012a2c -> 000161a0 -> 0002414f
3,4,8,00012a2c -> 000161a0 -> 0002414f -> 00046a56


**Висновок 3.3.** Обхід термінується на глибині **4** і дає саме ту структуру, що закладена
сідом: **99 424** кореня (99 441 клієнт мінус 17 referred), далі **2 → 4 → 8** вузли —
бінарне дерево. `sample_path` показує повний шлях від кореня, і його довжина завжди дорівнює
`depth`. Рекурсія зупинилась сама, без `depth guard`: `UNION ALL` перестав давати нові рядки,
коли на 4-му рівні в дерева закінчились діти.

Той самий обхід, але повні шляхи всередині дерева (`depth > 1`):

In [23]:
%%sql
WITH RECURSIVE referral_depth AS (
    SELECT
        c.customer_id,
        1                            AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        LEFT(c.customer_id, 8)::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    SELECT
        r.referred_customer_id,
        rd.depth + 1,
        rd.path_ids || r.referred_customer_id,
        rd.path_text || ' -> ' || LEFT(r.referred_customer_id, 8)
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
      ON rd.customer_id = r.referrer_customer_id
    WHERE NOT (r.referred_customer_id = ANY (rd.path_ids))
      AND rd.depth < 50
)
SELECT depth, LEFT(customer_id, 8) AS customer, path_text
FROM referral_depth
WHERE depth > 1
ORDER BY depth, path_text;

14 rows affected.

,depth,customer,path_text
0,2,000161a0,00012a2c -> 000161a0
1,2,0001fd61,00012a2c -> 0001fd61
2,3,0002414f,00012a2c -> 000161a0 -> 0002414f
3,3,000379cd,00012a2c -> 000161a0 -> 000379cd
4,3,0004164d,00012a2c -> 0001fd61 -> 0004164d
...,...,...,...
9,4,0005aefb,00012a2c -> 000161a0 -> 000379cd -> 0005aefb
10,4,00062b33,00012a2c -> 0001fd61 -> 0004164d -> 00062b33
11,4,00066ccb,00012a2c -> 0001fd61 -> 0004164d -> 00066ccb
12,4,00072d03,00012a2c -> 0001fd61 -> 000419c5 -> 00072d03


**Висновок 3.4.** **14** вузлів під коренями — рівно стільки, скільки ребер дерева, тобто
жоден шлях не продубльовано. Видно обидві гілки від кореня `00012a2c` і те, що шлях у
`path_text` накопичується саме так, як росте `path_ids`: він і є «журналом» відвіданих вузлів,
на який спирається захист від циклів.

### 3.2 Цикл не має кореня: що видно в охопленні

Перший бік захисту від циклів — не той, про який думають одразу. `NOT EXISTS`-anchor бере лише
вузли без вхідного ребра, а в замкненому циклі такого вузла немає: кожен його учасник комусь
referred. Тому вузли циклу **взагалі не досягаються** з коренів і тихо випадають зі звіту.
У production це теж баг, і ловити його треба явно — наприклад, таким запитом.

In [24]:
%%sql
WITH RECURSIVE referral_depth AS (
    SELECT c.customer_id, 1 AS depth, ARRAY[c.customer_id]::TEXT[] AS path_ids
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )
    UNION ALL
    SELECT r.referred_customer_id, rd.depth + 1, rd.path_ids || r.referred_customer_id
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd ON rd.customer_id = r.referrer_customer_id
    WHERE NOT (r.referred_customer_id = ANY (rd.path_ids))
      AND rd.depth < 50
)
SELECT
    (SELECT COUNT(*) FROM hw5_referrals)                  AS edges_total,
    (SELECT MAX(depth) FROM referral_depth)               AS max_depth,
    (SELECT COUNT(*) FROM referral_depth WHERE depth > 1) AS reached_from_roots,
    (SELECT COUNT(*) FROM hw5_referrals AS r
     WHERE NOT EXISTS (SELECT 1 FROM referral_depth AS rd
                       WHERE rd.customer_id = r.referred_customer_id)) AS unreachable_nodes;

1 rows affected.

,edges_total,max_depth,reached_from_roots,unreachable_nodes
0,17,4,14,3


**Висновок 3.5.** З 17 ребер із коренів досяжні **14**, а **3** вузли — недосяжні. Це рівно
учасники циклу. `NOT EXISTS`-anchor бере лише вузли без вхідного ребра, а в замкненому колі
такого вузла немає, тому цикл не має кореня і в основний звіт не потрапляє зовсім. Для
production це окремий клас багів: дані не «зламали» запит, вони з нього **тихо зникли**, і
ловити це треба саме такою перевіркою охоплення.

### 3.3 Що робить захист від циклів

Другий бік — власне termination. Запит нижче **навмисно** стартує обхід усередині циклу: за
anchor беруться якраз ті вузли, які попередня клітинка визнала недосяжними з коренів. Так
виглядала б ситуація в production, якби хтось додав ребро «назад» і корінь зник.

Без `path_ids` такий обхід крутився б вічно — PostgreSQL виконував би рекурсивний крок, поки не
закінчилась би памʼять під робочу таблицю. З перевіркою `NOT (next = ANY (path_ids))` кожен шлях
закінчується рівно тоді, коли наступний вузол уже є у власному шляху; колонка `blocked_next`
показує, на якому саме вузлі рекурсія спинилась. `depth < 50` — другий, незалежний запобіжник:
він зупинить обхід навіть на графі, де цикл побудований так, що перевірка шляху його не ловить
(наприклад, при помилці в самій умові).

In [25]:
%%sql
WITH RECURSIVE rooted AS (
    -- те саме дерево з коренів, що й у 3.1 — потрібне, щоб знайти недосяжні вузли
    SELECT c.customer_id, ARRAY[c.customer_id]::TEXT[] AS path_ids
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )
    UNION ALL
    SELECT r.referred_customer_id, ro.path_ids || r.referred_customer_id
    FROM hw5_referrals AS r
    JOIN rooted AS ro ON ro.customer_id = r.referrer_customer_id
    WHERE NOT (r.referred_customer_id = ANY (ro.path_ids))
),
cycle_nodes AS (
    -- вузли циклу: referred, але недосяжні з жодного кореня
    SELECT r.referred_customer_id AS customer_id
    FROM hw5_referrals AS r
    WHERE NOT EXISTS (
        SELECT 1 FROM rooted AS ro
        WHERE ro.customer_id = r.referred_customer_id
    )
),
walk AS (
    -- ANCHOR: примусово стартуємо всередині циклу
    SELECT
        cn.customer_id,
        1                             AS depth,
        ARRAY[cn.customer_id]::TEXT[] AS path_ids,
        LEFT(cn.customer_id, 8)::TEXT AS path_text
    FROM cycle_nodes AS cn

    UNION ALL

    -- RECURSIVE: той самий крок, із тими самими двома запобіжниками
    SELECT
        r.referred_customer_id,
        w.depth + 1,
        w.path_ids || r.referred_customer_id,
        w.path_text || ' -> ' || LEFT(r.referred_customer_id, 8)
    FROM hw5_referrals AS r
    JOIN walk AS w ON w.customer_id = r.referrer_customer_id
    WHERE NOT (r.referred_customer_id = ANY (w.path_ids))
      AND w.depth < 50
)
SELECT
    w.depth,
    w.path_text,
    (SELECT LEFT(r.referred_customer_id, 8)
     FROM hw5_referrals AS r
     WHERE r.referrer_customer_id = w.customer_id
       AND r.referred_customer_id = ANY (w.path_ids)) AS blocked_next
FROM walk AS w
ORDER BY w.path_text, w.depth;

9 rows affected.

,depth,path_text,blocked_next
0,1,000bf812,NaN
1,2,000bf812 -> 000e9434,NaN
2,3,000bf812 -> 000e9434 -> 000f17e2,000bf812
3,1,000e9434,NaN
4,2,000e9434 -> 000f17e2,NaN
5,3,000e9434 -> 000f17e2 -> 000bf812,000e9434
6,1,000f17e2,NaN
7,2,000f17e2 -> 000bf812,NaN
8,3,000f17e2 -> 000bf812 -> 000e9434,000f17e2


**Висновок 3.6.** Запит стартує з усіх трьох вузлів циклу, і кожен обхід зупиняється на
**depth = 3** — рівно тоді, коли наступним мав би стати вузол, який уже є у власному шляху.
Колонка `blocked_next` називає цей вузол: для старту `000bf812` рекурсія спинилась перед
повторним `000bf812`. Без `path_ids` кожна з цих трьох гілок крутилася б вічно, а `depth < 50`
обрізав би її на 50-му кроці — тобто depth guard дав би термінацію, але **неправильний
результат** (50 рядків замість 3). Саме тому потрібні обидва запобіжники: перевірка шляху
відповідає за коректність, depth guard — за те, що запит узагалі завершиться.

---

## Завдання 4. Materialized View із CONCURRENT REFRESH

`MATERIALIZED VIEW` містить **той самий повний pipeline** із Завдання 2, а не скорочену версію:
нижче наведено весь CTE-chain цілком. Автоперевірка в кінці ноутбука порівнює MV з живим
pipeline через `EXCEPT` в обидва боки — якщо колись розійдуться, клітинка впаде.

`UNIQUE INDEX` на `customer_id` — не оптимізація, а **умова роботи**
`REFRESH MATERIALIZED VIEW CONCURRENTLY`: без унікального, повного (не partial) індексу
PostgreSQL відмовиться робити неблокуючий refresh.

In [26]:
%%sql
CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: стабільна точка відліку recency — з самих даних, а не CURRENT_DATE.
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: грануляність «одне замовлення — один рядок»: позиції + доставка.
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp       AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2)
                                         AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
           ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM — recency_days від as_of_date, frequency, monetary, перша покупка.
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*)                                     AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2)           AS monetary,
        MIN(s.order_date)                            AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: квартал когорти за датою першого замовлення.
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day AOV без втрати деталізації рядків + маркер останнього замовлення.
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: частка скасованих замовлень клієнта.
    SELECT
        customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4)
            AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: виручка в розрізі клієнт x категорія товару.
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2)
            AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
      ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
           ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: top-1 категорія за виручкою; нічию розвʼязує назва категорії.
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_unique_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999)                     AS recency_days,
    COALESCE(coh.frequency, 0)                           AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2)            AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d,
    COALESCE(cx.cancellation_rate, 0)::NUMERIC(6, 4)     AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)')         AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2)     AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
       ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
       ON aov.customer_id = c.customer_id
      AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cx
       ON cx.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
       ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);

99441 rows affected.

""


`REFRESH ... CONCURRENTLY` не можна виконати в транзакції, а JupySQL кожну клітинку виконує
саме в ній. Тому ця команда йде через окреме зʼєднання з `isolation_level='AUTOCOMMIT'`.
Для порівняння тут же виміряно звичайний блокуючий `REFRESH`.

In [27]:
def run_autocommit(sql):
    """REFRESH ... CONCURRENTLY вимагає виконання поза транзакційним блоком."""
    with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
        conn.exec_driver_sql(sql)


def timed(sql, label):
    started = time.perf_counter()
    run_autocommit(sql)
    elapsed = time.perf_counter() - started
    print(f"{label:52s} {elapsed:6.2f} с")
    return elapsed


t_plain = timed("REFRESH MATERIALIZED VIEW mv_olist_customer_features",
                "REFRESH MATERIALIZED VIEW (блокуючий)")
t_concurrent = timed("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features",
                     "REFRESH MATERIALIZED VIEW CONCURRENTLY")
print(f"\nCONCURRENTLY дорожчий у {t_concurrent / t_plain:.2f} раза — "
      "платимо за те, що читачі не блокуються.")

REFRESH MATERIALIZED VIEW (блокуючий)                  2.62 с


REFRESH MATERIALIZED VIEW CONCURRENTLY                 3.14 с

CONCURRENTLY дорожчий у 1.20 раза — платимо за те, що читачі не блокуються.


Розмір і вміст MV: рядків стільки ж, скільки клієнтів, і рівно один рядок на `customer_id`.

In [28]:
%%sql
SELECT
    (SELECT COUNT(*) FROM mv_olist_customer_features)              AS mv_rows,
    (SELECT COUNT(DISTINCT customer_id) FROM mv_olist_customer_features) AS mv_customers,
    (SELECT COUNT(*) FROM olist_customers)                          AS source_customers,
    pg_size_pretty(pg_total_relation_size('mv_olist_customer_features')) AS mv_size,
    pg_size_pretty(pg_total_relation_size('olist_orders')
                 + pg_total_relation_size('olist_order_items'))     AS source_size;

1 rows affected.

,mv_rows,mv_customers,source_customers,mv_size,source_size
0,99441,99441,99441,22 MB,46 MB


**Висновок 4.1.** MV містить **99 441** рядок — рівно по одному на `customer_id`, грануляція
не роздулась. Кеш займає **22 MB** проти **46 MB** вихідних `olist_orders` +
`olist_order_items`: ознаки щільніші за сирі дані, але це все одно додаткові 22 MB, які треба
зберігати й оновлювати. Це і є storage-cost у компромісі, який розбирається в Reflection.

In [29]:
%%sql
SELECT customer_id, customer_state, recency_days, frequency, monetary,
       cohort_quarter, avg_order_value_30d, cancellation_rate, preferred_category
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;

5 rows affected.

,customer_id,customer_state,recency_days,frequency,monetary,cohort_quarter,avg_order_value_30d,cancellation_rate,preferred_category
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-07-01,13664.08,0.0000,telefonia_fixa
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-01,7274.88,0.0000,telefonia_fixa
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-01-01,6929.31,0.0000,utilidades_domesticas
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-01,6922.21,0.0000,pcs
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-04-01,6726.66,0.0000,artes


**Висновок 4.2.** Топ-5 за `monetary` збігається з виводом live-pipeline із розділу 2 рядок у
рядок — MV справді містить той самий повний pipeline, а не його скорочену версію. Автоперевірка
в кінці ноутбука підтверджує це формально, через `EXCEPT` в обидва боки.

### 4.1 Freshness: що саме означає «застарілі ознаки»

MV — це знімок. Клітинка нижче вставляє одне нове замовлення (`hw5_probe_order`), показує, що
ознаки клієнта в MV **не змінилися**, робить `REFRESH CONCURRENTLY`, показує оновлені значення —
і прибирає probe-рядок, повертаючи базу у вихідний стан. Дата probe-замовлення взята в середині
історії, щоб не зсунути `reference_date` і не переписати `recency_days` усім клієнтам.

In [30]:
from IPython import get_ipython

ip = get_ipython()


def q(sql_text):
    """Виконує SQL через JupySQL і повертає результат як DataFrame."""
    return ip.run_cell_magic("sql", "", sql_text)


probe_customer = q("""
    SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1;
""").iloc[0, 0]

features_sql = f"""
    SELECT frequency, monetary, recency_days
    FROM {{table}}
    WHERE customer_id = '{probe_customer}';
"""


def snapshot(stage):
    mv = q(features_sql.format(table="mv_olist_customer_features")).iloc[0]
    live = q(features_sql.format(table="v_olist_customer_features")).iloc[0]
    return {"stage": stage,
            "mv_frequency": mv["frequency"], "live_frequency": live["frequency"],
            "mv_monetary": mv["monetary"], "live_monetary": live["monetary"]}


states = [snapshot("1. початковий стан")]

q(f"""
    INSERT INTO olist_orders (order_id, customer_id, order_status,
                              order_purchase_timestamp, order_approved_at)
    VALUES ('hw5_probe_order', '{probe_customer}', 'delivered',
            TIMESTAMP '2018-01-15 12:00:00', TIMESTAMP '2018-01-16 12:00:00');
""")
states.append(snapshot("2. після INSERT, до REFRESH"))

run_autocommit("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features")
states.append(snapshot("3. після REFRESH CONCURRENTLY"))

q("DELETE FROM olist_orders WHERE order_id = 'hw5_probe_order';")
run_autocommit("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features")
states.append(snapshot("4. probe прибрано, MV оновлена"))

print("Клієнт:", probe_customer)
pd.DataFrame(states)

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

...

1 rows affected.

1 rows affected.

1 rows affected.

Клієнт: 00012a2ce6f8dcda20d059ce98491703


,stage,mv_frequency,live_frequency,mv_monetary,live_monetary
0,1. початковий стан,1,1,114.74,114.74
1,"2. після INSERT, до REFRESH",1,2,114.74,114.74
2,3. після REFRESH CONCURRENTLY,2,2,114.74,114.74
3,"4. probe прибрано, MV оновлена",1,1,114.74,114.74


**Висновок 4.4.** Чотири стани показують весь життєвий цикл staleness. Після `INSERT` live-view
уже бачить `frequency = 2`, а MV усе ще віддає **1** — модель, яка читає з кешу, працювала б на
застарілій ознаці. `REFRESH CONCURRENTLY` вирівнює обидві колонки на **2**, а після прибирання
probe-рядка й повторного refresh усе повертається до **1**: клітинка ідемпотентна, тому
*Restart & Run All* дає той самий результат.

`monetary` при цьому не змінився (**114.74** в усіх чотирьох станах) — probe-замовлення не має
позицій, тому `order_total = 0`. Це корисна деталь: вікно застарівання різне для різних ознак,
і «MV застаріла» не означає «всі колонки поїхали».

---

## Завдання 5. EXPLAIN (ANALYZE, BUFFERS)

Порівнюємо три сценарії читання **одного й того самого** набору ознак:

1. повне переобчислення CTE-pipeline (те, що робить `VIEW` при кожному запиті);
2. lookup одного клієнта в MV за `customer_id` — має піти по `UNIQUE INDEX`;
3. top-10 за `monetary` з MV — до і після створення індексу на `monetary`.

Плани друкуємо рядками: `pandas` обрізав би їх за `display.max_rows`.

In [31]:
import re

PIPELINE_SQL = """
WITH
reference_date AS (
    -- Stage 0: стабільна точка відліку recency — з самих даних, а не CURRENT_DATE.
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: грануляність «одне замовлення — один рядок»: позиції + доставка.
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp       AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2)
                                         AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
           ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM — recency_days від as_of_date, frequency, monetary, перша покупка.
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*)                                     AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2)           AS monetary,
        MIN(s.order_date)                            AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: квартал когорти за датою першого замовлення.
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day AOV без втрати деталізації рядків + маркер останнього замовлення.
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: частка скасованих замовлень клієнта.
    SELECT
        customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4)
            AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: виручка в розрізі клієнт x категорія товару.
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2)
            AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
      ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
           ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: top-1 категорія за виручкою; нічию розвʼязує назва категорії.
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_unique_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999)                     AS recency_days,
    COALESCE(coh.frequency, 0)                           AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2)            AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d,
    COALESCE(cx.cancellation_rate, 0)::NUMERIC(6, 4)     AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)')         AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2)     AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
       ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
       ON aov.customer_id = c.customer_id
      AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cx
       ON cx.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
       ON pc.customer_id = c.customer_id
"""


def explain(sql, label):
    """Друкує план і повертає зведення: таймінги й сумарні buffers по вузлах."""
    plan = pd.read_sql_query("EXPLAIN (ANALYZE, BUFFERS) " + sql, engine).iloc[:, 0].tolist()
    print("=" * 100)
    print(label)
    print("=" * 100)
    print("\n".join(plan))
    print()

    def grab(prefix):
        for line in plan:
            if line.strip().startswith(prefix):
                return float(re.search(r"([\d.]+)\s*ms", line).group(1))
        return float("nan")

    hit = read = 0
    for line in plan:
        found = re.search(r"shared hit=(\d+)(?:\s+read=(\d+))?", line)
        if found:
            hit += int(found.group(1))
            read += int(found.group(2) or 0)
    return {"сценарій": label, "planning_ms": grab("Planning Time"),
            "execution_ms": grab("Execution Time"), "shared_hit": hit, "shared_read": read}


timings = []
timings.append(explain(PIPELINE_SQL, "1. Повне переобчислення CTE-pipeline"))

1. Повне переобчислення CTE-pipeline
Hash Left Join  (cost=81668.55..98291.38 rows=84100 width=190) (actual time=1231.417..1797.389 rows=99441 loops=1)
  Hash Cond: (c.customer_id = ranked.customer_id)
  Buffers: shared hit=12451, temp read=11370 written=11524
  CTE stage1_orders_total
    ->  Finalize GroupAggregate  (cost=16727.31..27528.63 rows=88641 width=124) (actual time=97.972..213.815 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=4159, temp read=1545 written=1549
          ->  Gather Merge  (cost=16727.31..25490.56 rows=64913 width=136) (actual time=97.957..171.049 rows=99661 loops=1)
                Workers Planned: 1
                Workers Launched: 1
                Buffers: shared hit=4159, temp read=1545 written=1549
                ->  Partial GroupAggregate  (cost=15727.30..17187.84 rows=64913 width=136) (actual time=82.666..126.306 rows=49831 loops=2)
                      Group Key: o_1.order_id
                      Buffers: shar

**Висновок 5.1.** Повне переобчислення — **1951 мс** і **140 469** звернень до буферів. У плані
видно, звідки саме час:

| Гілка плану | Stage | `actual time` |
|---|---|---|
| `Sort` (`external merge  Disk: 9440kB`) → `WindowAgg` → `Incremental Sort` → `WindowAgg` | `stage4_rolling_aov` | 520.9 → **724.9 мс** |
| `Sort` (`Disk: 4104kB`) → `Finalize GroupAggregate` → `WindowAgg` | `stage6` + `stage7` | 207.9 → **381.9 мс** |
| `Finalize GroupAggregate` над `Parallel Hash Right Join` | `stage1_orders_total` | 98.0 → **213.8 мс** |
| `HashAggregate` (`Batches: 21  Disk Usage: 7400kB`) | `stage2_rfm` | 57.2 → **113.2 мс** |
| `HashAggregate` (`Batches: 21  Disk Usage: 7192kB`) | `stage5_cancel_rate` | 37.3 → **90.1 мс** |
| верхній `Hash Left Join`, що зшиває все на 99 441 клієнта | фінальний `SELECT` | 1231.4 → **1797.4 мс** |

Два спостереження. По-перше, `stage1_orders_total` у плані — це `CTE Scan`, тобто PostgreSQL
матеріалізував його **один раз** і просканував чотири рази (для `stage2`, `stage4`, `stage5` і
фінального join-у). Саме тому CTE-chain тут не дорожчий за один великий запит. По-друге, майже
всі важкі вузли **спілять на диск** (`external merge`, `Disk Usage`) — робоча памʼять за
замовчуванням не вміщає 99 тис. рядків, і це головна причина, чому йдеться про секунди, а не
про десятки мілісекунд.

In [32]:
timings.append(explain("""
    SELECT *
    FROM mv_olist_customer_features
    WHERE customer_id = (SELECT MIN(customer_id) FROM olist_customers)
""", "2. Lookup одного клієнта в MV за UNIQUE INDEX"))

2. Lookup одного клієнта в MV за UNIQUE INDEX
Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features  (cost=0.98..8.99 rows=1 width=190) (actual time=0.109..0.110 rows=1 loops=1)
  Index Cond: (customer_id = $1)
  Buffers: shared hit=8
  InitPlan 2 (returns $1)
    ->  Result  (cost=0.55..0.56 rows=1 width=32) (actual time=0.085..0.085 rows=1 loops=1)
          Buffers: shared hit=4
          InitPlan 1 (returns $0)
            ->  Limit  (cost=0.42..0.55 rows=1 width=32) (actual time=0.082..0.083 rows=1 loops=1)
                  Buffers: shared hit=4
                  ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..11084.82 rows=83680 width=32) (actual time=0.081..0.082 rows=1 loops=1)
                        Index Cond: (customer_id IS NOT NULL)
                        Heap Fetches: 1
                        Buffers: shared hit=4
Planning:
  Buffers: shared hit=10
Planning Time: 0.792 ms
Execution Time: 0.127 ms



**Висновок 5.2.** Lookup одного клієнта — `Index Scan using
idx_mv_olist_customer_features_customer_id` з `Index Cond: (customer_id = $1)`: **0.127 мс** і
**8** буферних читань по самому MV. Жодного `Sort`, `HashAggregate` чи `WindowAgg` у плані
немає — вся робота pipeline уже зроблена під час `REFRESH`. `InitPlan` для
`MIN(customer_id)` теж іде по індексу (`Index Only Scan using olist_customers_pkey`,
`Heap Fetches: 1`), тому разом із плануванням запит залишається в межах однієї мілісекунди.

Третій сценарій — top-N читання. Спочатку без індексу на `monetary`, тому планувальник мусить
відсортувати весь MV; потім той самий запит з індексом `(monetary DESC)`.

In [33]:
TOP_N_SQL = """
    SELECT customer_id, monetary, frequency, recency_days
    FROM mv_olist_customer_features
    ORDER BY monetary DESC NULLS LAST
    LIMIT 10
"""

timings.append(explain(TOP_N_SQL, "3. Top-10 за monetary БЕЗ індексу"))

run_autocommit("""
    CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
        ON mv_olist_customer_features (monetary DESC NULLS LAST)
""")
run_autocommit("ANALYZE mv_olist_customer_features")

timings.append(explain(TOP_N_SQL, "4. Top-10 за monetary З індексом (monetary DESC)"))

3. Top-10 за monetary БЕЗ індексу
Limit  (cost=4961.01..4962.16 rows=10 width=60) (actual time=17.128..19.492 rows=10 loops=1)
  Buffers: shared hit=2148
  ->  Gather Merge  (cost=4961.01..11687.94 rows=58495 width=60) (actual time=17.127..19.489 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=2148
        ->  Sort  (cost=3961.00..4107.24 rows=58495 width=60) (actual time=9.116..9.117 rows=9 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=2148
              Worker 0:  Sort Method: top-N heapsort  Memory: 27kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2696.95 rows=58495 width=60) (actual time=0.008..4.190 rows=49721 loops=2)
                    Buffers: shared hit=2112
Planning Time: 0.045 ms
Execution Time: 19.505 ms



4. Top-10 за monetary З індексом (monetary DESC)
Limit  (cost=0.42..1.53 rows=10 width=51) (actual time=0.020..0.030 rows=10 loops=1)
  Buffers: shared hit=10 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary on mv_olist_customer_features  (cost=0.42..11044.01 rows=99441 width=51) (actual time=0.019..0.028 rows=10 loops=1)
        Buffers: shared hit=10 read=3
Planning:
  Buffers: shared hit=16 read=1
Planning Time: 0.120 ms
Execution Time: 0.038 ms



**Висновок 5.3.** Top-N показує, що MV сама по собі ще не робить запит швидким — потрібен
правильний індекс. Без індексу план — `Limit` → `Gather Merge` → `Sort` (`top-N heapsort`) →
`Parallel Seq Scan on mv_olist_customer_features`: **19.5 мс** і **8 556** буферів, тобто
прочитано весь MV, щоб віддати 10 рядків. З індексом `(monetary DESC NULLS LAST)` план
згортається до `Limit` → `Index Scan using idx_mv_olist_customer_features_monetary`:
**0.038 мс** і **43** буфери. Важлива деталь — `NULLS LAST` в індексі має збігатися з
`ORDER BY`, інакше планувальник індекс не візьме.

Зведення по чотирьох планах — саме ці числа цитуються у Reflection:

In [34]:
summary = pd.DataFrame(timings)
summary["total_ms"] = (summary["planning_ms"] + summary["execution_ms"]).round(3)
summary["швидше за pipeline"] = (
    summary.loc[0, "execution_ms"] / summary["execution_ms"]
).round(1).astype(str) + "x"
summary

,сценарій,planning_ms,execution_ms,shared_hit,shared_read,total_ms,швидше за pipeline
0,1. Повне переобчислення CTE-pipeline,0.641,1951.246,140469,0,1951.887,1.0x
1,2. Lookup одного клієнта в MV за UNIQUE INDEX,0.792,0.127,30,0,0.919,15364.1x
2,3. Top-10 за monetary БЕЗ індексу,0.045,19.505,8556,0,19.550,100.0x
3,4. Top-10 за monetary З індексом (monetary DESC),0.120,0.038,36,7,0.158,51348.6x


**Висновок 5.4.** Зведення по чотирьох сценаріях читання одного й того самого набору ознак:

| Сценарій | Execution | Буфери | Проти pipeline |
|---|---|---|---|
| повне переобчислення CTE-pipeline | 1951.2 мс | 140 469 | 1x |
| top-10 з MV без індексу | 19.5 мс | 8 556 | 100x |
| lookup за `customer_id` (UNIQUE INDEX) | 0.127 мс | 30 | 15 364x |
| top-10 з MV з індексом на `monetary` | 0.038 мс | 43 | 51 349x |

Таймінги трохи змінюються між прогонами (кеш, паралельні воркери), але порядки стабільні:
матеріалізація дає два порядки, матеріалізація плюс доречний індекс — чотири.

---

## 5.5 Reflection

**Найдорожча ознака — rolling 30-day AOV.** У плані її гілка найдовша: `Sort` із
`Sort Method: external merge  Disk: 9440kB`, над ним `WindowAgg`, `Incremental Sort` і другий
`WindowAgg` — від 520.9 до **724.9 мс**. Дороге не усереднення, а сортування: `RANGE BETWEEN
INTERVAL '30 days' PRECEDING` вимагає впорядкувати кожну партицію, і 99 441 рядок не вміщається
в `work_mem`, тому сортування йде на диск. Друга за вартістю — `preferred_category`
(`GroupAggregate` + `WindowAgg`, 207.9→381.9 мс), бо вона вдруге джойнить `order_items` з
`products`. `HashAggregate` для RFM і cancellation rate дешевші (113 і 90 мс), хоч теж спілять
(`Batches: 21  Disk Usage: 7400kB`). Разом — **1951 мс** і **140 469** буферів.

**Кешувати варто.** Lookup у MV — `Index Scan using idx_mv_olist_customer_features_customer_id`,
**0.127 мс** і 8 буферів: у **15 364 раза** швидше. `REFRESH CONCURRENTLY` коштує 3.14 с проти
2.62 с блокуючого — 20 % надбавки за те, що читачі не чекають. Цикл я б узяв **щодобовий**:
`cohort_quarter` не змінюється ніколи, `recency_days` — раз на добу, тож ганяти pipeline частіше
нема за чим.

**Звичайний `VIEW`** доречний там, де є `v_olist_customer_features`: він безкоштовний за
storage і завжди свіжий. Якби ознаки читались раз на тиждень для звіту або джерело мінялось
щохвилини, 22 МБ кешу й цикл refresh були б чистими витратами.

**Termination** тримається на двох запобіжниках. `NOT (next = ANY (path_ids))` дає коректність:
при циклі `A → B → A` вузол `A` уже у `path_ids` на кроці з `B`, рядок не народжується, і обхід
спиняється на `depth = 3` (розділ 3.6). `depth < 50` — страховка на випадок помилки в першій
умові; без обох запит генерував би рядки, доки не впаде по памʼяті. Декларативна альтернатива —
`CYCLE`-clause PostgreSQL 14+.

**Головний trade-off — recompute-cost.** Дані історичні, тому freshness майже нічого не варта, а
22 МБ проти 46 МБ джерела — дешево. Неприйнятні саме 2 секунди на кожне звернення.

---

## Автоперевірки

Клітинка падає, якщо будь-який обовʼязковий артефакт ДЗ відсутній або розійшовся з очікуваним:
від рівності raw/typed до того, що MV — це справді повний pipeline, а не його скорочена версія.

In [35]:
# --- 1. raw і typed рівні збігаються рядок у рядок -----------------------
counts = q("""
    SELECT
        (SELECT COUNT(*) FROM olist_customers_raw)   AS raw_customers,
        (SELECT COUNT(*) FROM olist_customers)       AS typed_customers,
        (SELECT COUNT(*) FROM olist_orders_raw)      AS raw_orders,
        (SELECT COUNT(*) FROM olist_orders)          AS typed_orders,
        (SELECT COUNT(*) FROM olist_order_items_raw) AS raw_items,
        (SELECT COUNT(*) FROM olist_order_items)     AS typed_items,
        (SELECT COUNT(*) FROM olist_products_raw)    AS raw_products,
        (SELECT COUNT(*) FROM olist_products)        AS typed_products,
        (SELECT COUNT(*) FROM olist_sellers_raw)     AS raw_sellers,
        (SELECT COUNT(*) FROM olist_sellers)         AS typed_sellers;
""").iloc[0]
for name in ("customers", "orders", "items", "products", "sellers"):
    assert counts[f"raw_{name}"] == counts[f"typed_{name}"], f"raw != typed для {name}"

# --- 2. обмеження typed-схеми справді існують ----------------------------
cons = q("""
    SELECT
        COUNT(*) FILTER (WHERE con.contype = 'p') AS pk,
        COUNT(*) FILTER (WHERE con.contype = 'f') AS fk,
        COUNT(*) FILTER (WHERE con.contype = 'c') AS checks
    FROM pg_constraint AS con
    JOIN pg_class     AS c ON c.oid = con.conrelid
    JOIN pg_namespace AS n ON n.oid = c.relnamespace
    WHERE n.nspname = 'public'
      AND c.relname IN ('olist_customers', 'olist_orders', 'olist_order_items',
                        'olist_products', 'olist_sellers');
""").iloc[0]
assert cons["pk"] == 5, f"очікували 5 PRIMARY KEY, маємо {cons['pk']}"
assert cons["fk"] == 4, f"очікували 4 FOREIGN KEY, маємо {cons['fk']}"
assert cons["checks"] >= 10, f"замало CHECK: {cons['checks']}"

# --- 3. усі обовʼязкові ознаки присутні як колонки MV --------------------
# information_schema.columns не показує MATERIALIZED VIEW — лише таблиці й звичайні
# view, тому колонки беремо з pg_attribute.
mv_columns = set(q("""
    SELECT a.attname AS column_name
    FROM pg_attribute AS a
    JOIN pg_class     AS c ON c.oid = a.attrelid
    WHERE c.relname = 'mv_olist_customer_features'
      AND a.attnum > 0
      AND NOT a.attisdropped;
""")["column_name"])
required_features = {"recency_days", "frequency", "monetary",          # RFM
                     "cohort_quarter", "avg_order_value_30d",
                     "cancellation_rate", "preferred_category"}
assert required_features <= mv_columns, f"немає ознак: {required_features - mv_columns}"

# --- 4. MV — це повний pipeline, а не скорочена версія -------------------
drift = q("""
    SELECT
        (SELECT COUNT(*) FROM (
            SELECT * FROM mv_olist_customer_features
            EXCEPT SELECT * FROM v_olist_customer_features) AS d) AS mv_minus_live,
        (SELECT COUNT(*) FROM (
            SELECT * FROM v_olist_customer_features
            EXCEPT SELECT * FROM mv_olist_customer_features) AS d) AS live_minus_mv;
""").iloc[0]
assert drift["mv_minus_live"] == 0 and drift["live_minus_mv"] == 0, \
    f"MV розійшлася з pipeline: {dict(drift)}"

# --- 5. грануляність MV: рівно один рядок на клієнта ---------------------
grain = q("""
    SELECT
        (SELECT COUNT(*) FROM mv_olist_customer_features)                    AS mv_rows,
        (SELECT COUNT(DISTINCT customer_id) FROM mv_olist_customer_features) AS mv_customers,
        (SELECT COUNT(*) FROM olist_customers)                               AS customers;
""").iloc[0]
assert grain["mv_rows"] == grain["mv_customers"] == grain["customers"], \
    f"MV не на рівні customer_id: {dict(grain)}"

# --- 6. UNIQUE INDEX існує, унікальний і не partial ----------------------
idx = q("""
    SELECT i.indisunique, i.indpred IS NULL AS is_full_index
    FROM pg_index AS i
    JOIN pg_class AS c ON c.oid = i.indexrelid
    WHERE c.relname = 'idx_mv_olist_customer_features_customer_id';
""")
assert len(idx) == 1 and bool(idx.iloc[0]["indisunique"]) and bool(idx.iloc[0]["is_full_index"]), \
    "немає повного UNIQUE INDEX на customer_id — REFRESH CONCURRENTLY був би неможливий"

# --- 7. recursive CTE: шляхи без повторів, цикл не роздув обхід ----------
rec = q("""
    WITH RECURSIVE referral_depth AS (
        SELECT c.customer_id, 1 AS depth, ARRAY[c.customer_id]::TEXT[] AS path_ids
        FROM olist_customers AS c
        WHERE NOT EXISTS (SELECT 1 FROM hw5_referrals AS r
                          WHERE r.referred_customer_id = c.customer_id)
        UNION ALL
        SELECT r.referred_customer_id, rd.depth + 1, rd.path_ids || r.referred_customer_id
        FROM hw5_referrals AS r
        JOIN referral_depth AS rd ON rd.customer_id = r.referrer_customer_id
        WHERE NOT (r.referred_customer_id = ANY (rd.path_ids))
          AND rd.depth < 50
    ),
    deep AS (SELECT * FROM referral_depth WHERE depth > 1)
    SELECT
        (SELECT MAX(depth) FROM referral_depth) AS max_depth,
        (SELECT COUNT(*) FROM deep)             AS reached,
        (SELECT COUNT(*) FROM deep
          WHERE ARRAY_LENGTH(path_ids, 1) <> depth) AS broken_paths,
        (SELECT COUNT(*) FROM deep AS d
          WHERE (SELECT COUNT(DISTINCT x) FROM UNNEST(d.path_ids) AS x) <> d.depth)
                                                AS paths_with_duplicates;
""").iloc[0]
assert rec["broken_paths"] == 0, "довжина path_ids не відповідає depth"
assert rec["paths_with_duplicates"] == 0, "вузол зустрічається у власному шляху двічі"
assert rec["max_depth"] == 4, f"очікували дерево глибиною 4, маємо {rec['max_depth']}"
assert rec["reached"] == 14, f"очікували 14 вузлів під коренями, маємо {rec['reached']}"

# --- 8. probe-рядок прибрано, база у вихідному стані ---------------------
probe_left = q("""
    SELECT COUNT(*) AS n FROM olist_orders WHERE order_id = 'hw5_probe_order';
""").iloc[0]["n"]
assert probe_left == 0, "probe-замовлення залишилось у даних"

print("Усі автоперевірки пройдено.")
print(f"  typed-схема: {int(cons['pk'])} PK, {int(cons['fk'])} FK, {int(cons['checks'])} CHECK")
print(f"  feature store: {int(grain['mv_rows']):,} рядків, {len(mv_columns)} колонок")
print(f"  referral-дерево: глибина {int(rec['max_depth'])}, {int(rec['reached'])} вузлів під коренями")

1 rows affected.

1 rows affected.

12 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

Усі автоперевірки пройдено.
  typed-схема: 5 PK, 4 FK, 10 CHECK
  feature store: 99,441 рядків, 12 колонок
  referral-дерево: глибина 4, 14 вузлів під коренями


---

## Підсумки

| Етап | Артефакт у ноутбуці | Результат |
|---|---|---|
| 1. Setup і typed-схема | `pgserver` + PostgreSQL 16.2, `olist_*_raw` через `COPY`, 5 typed-таблиць | 99 441 / 112 650 / 32 951 / 3 095 рядків, `diff = 0`, 5 PK + 4 FK + 10 CHECK |
| 2. CTE-chain | `v_olist_customer_features`, 8 named stages | 99 441 рядок, 12 колонок, усі 5 обовʼязкових груп ознак |
| 3. Recursive CTE | `hw5_referrals` + обхід із `path_ids` і depth guard | дерево глибиною 4, 14 вузлів під коренями, цикл спинено на `depth = 3` |
| 4. Materialized View | `mv_olist_customer_features` + `UNIQUE INDEX` | 22 MB, `REFRESH CONCURRENTLY` 3.14 с, демонстрація staleness |
| 5. EXPLAIN + Reflection | 4 плани `(ANALYZE, BUFFERS)` + зведення | 1951 мс → 0.127 мс (15 364x) |
| Автоперевірки | 8 блоків `assert` | усі пройдено, MV ідентична live-pipeline (`EXCEPT` = 0 в обидва боки) |

Три речі, які в цій роботі виявились важливішими за синтаксис CTE.

**Грануляність ключа визначає, чи має ознака сенс.** На `customer_id` pipeline формально
правильний, але `frequency ≡ 1`, бо Olist видає новий `customer_id` на кожне замовлення. Поки
не порахуєш `COUNT(DISTINCT customer_unique_id)`, цього у виводі не видно — а модель на таких
ознаках просто не навчиться.

**Вартість pipeline — це сортування, а не агрегація.** Найдорожчий вузол плану — не `JOIN` і не
`HashAggregate`, а `Sort` під window-функцією, який не вміщається в `work_mem` і йде на диск.
Матеріалізація прибирає саме цю роботу, тому й дає два порядки швидкості.

**Захист від циклів потрібен у двох напрямках.** `path_ids` зупиняє обхід, який у цикл зайшов, а
перевірка охоплення (розділ 3.5) ловить протилежний випадок: цикл без кореня не ламає запит — він
**тихо зникає** з результату, і без явної перевірки цього не помітно.